In [ ]:

# Step 28: NLP Entity Extraction
# HanuAi ML Assessment - Text Mining

import re
import pandas as pd

# 1. Define component recognition patterns
component_patterns = {
    "Radio / Head Unit": r"\b(?:radio|stereo|head[\s-]*unit)\b",
    "Display / Screen": r"\b(?:screen|display|pantalla)\b",
    "Amplifier": r"\b(?:amplifier|amplificador)\b",
    "Camera": r"\b(?:camera|c[aá]mara)\b",
    "Bluetooth / Wi-Fi": r"\b(?:bluetooth|wi[\s-]?fi|hotspot)\b",
    "OnStar / Telematics": r"\b(?:onstar|telematics)\b",
    "SD Card / Reader": r"\b(?:sd[\s-]*card|card[\s-]*reader)\b",
    "Battery": r"\b(?:battery|bater[ií]a)\b",
    "Antenna / Coax": r"\b(?:antenna|coax(?:ial)?|antena)\b"
}

# 2. Define symptom recognition patterns
symptom_patterns = {
    "Black / Blank Screen":
        r"\b(?:black|blank)\s+(?:screen|display)\b|"
        r"\b(?:screen|display)\s+(?:goes?\s+)?(?:black|blank)\b|"
        r"\bpantalla\s+(?:se\s+pone\s+)?negra\b",

    "No Audio":
        r"\b(?:no|lost|without)\s+(?:audio|sound|sonido)\b",

    "Intermittent":
        r"\b(?:intermittent(?:ly)?|intermittantly|occasionally)\b",

    "Inoperative":
        r"\b(?:inop(?:erable)?|not\s+working|"
        r"does\s+not\s+work|unresponsive|irresponsive)\b",

    "Connection Failure":
        r"\b(?:connection\s+failure|lost\s+communication|"
        r"lost\s+satellite\s+signal|disconnect(?:ed|ing|s)?)\b",

    "Unresponsive Controls":
        r"\b(?:button|buttons|knob|knobs|touchscreen)\b"
        r".{0,30}\b(?:not\s+working|inop|unresponsive)\b"
}

# 3. Function to extract matching entity tags
def extract_tags(text, patterns):
    return [
        label
        for label, pattern in patterns.items()
        if re.search(pattern, str(text), flags=re.IGNORECASE)
    ]

# 4. Use customer and technician text for components
diagnostic_text = (
    df_clean["CUSTOMER_VERBATIM_CLEAN"].fillna("")
    + " "
    + df_clean["CAUSAL_VERBATIM_CLEAN"].fillna("")
)

# Use customer descriptions for reported symptoms
customer_text = df_clean["CUSTOMER_VERBATIM_CLEAN"].fillna("")

# 5. Extract components and symptoms
df_clean["NLP_Components"] = diagnostic_text.apply(
    lambda text: extract_tags(text, component_patterns)
)

df_clean["NLP_Symptoms"] = customer_text.apply(
    lambda text: extract_tags(text, symptom_patterns)
)

# 6. Extract standard 5-character diagnostic trouble codes
dtc_pattern = re.compile(
    r"(?<![A-Z0-9])[PCBU][0-3][0-9A-F]{3}(?![A-Z0-9])"
)

df_clean["NLP_Error_Codes"] = diagnostic_text.apply(
    lambda text: sorted(set(dtc_pattern.findall(text.upper())))
)

# 7. Display extracted entities
print("HANUAI - NLP ENTITY EXTRACTION RESULTS")

display(df_clean[
    [
        "Event id",
        "NLP_Components",
        "NLP_Symptoms",
        "NLP_Error_Codes"
    ]
].head(10))

# 8. Check extraction coverage
print("\nTOTAL RECORDS:", len(df_clean))

for col in [
    "NLP_Components",
    "NLP_Symptoms",
    "NLP_Error_Codes"
]:
    print(col, "- Records with tags:",
          df_clean[col].apply(bool).sum())


In [ ]:

# Step 27: Text Preprocessing for NLP
# HanuAi ML Assessment - Text Mining

import re
import html
import unicodedata
import pandas as pd

# Function to clean unstructured text
def preprocess_text(text):

    # Handle missing values
    if pd.isna(text):
        return ""

    # Convert input into text
    text = str(text)

    # Convert HTML entities into readable characters
    text = html.unescape(text)

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Remove invisible zero-width characters
    text = re.sub(r"[\u200b-\u200d\ufeff]", "", text)

    # Replace unwanted control characters with spaces
    text = re.sub(
        r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]",
        " ",
        text
    )

    # Replace repeated whitespace with one space
    text = re.sub(r"\s+", " ", text)

    # Remove leading and trailing spaces
    return text.strip()


# Create cleaned copies of all three text columns
for col in text_columns:
    df_clean[col + "_CLEAN"] = (
        df_clean[col].apply(preprocess_text)
    )

# Display original and cleaned customer complaints
print("TEXT PREPROCESSING RESULTS")

display(
    df_clean[
        ["CUSTOMER_VERBATIM", "CUSTOMER_VERBATIM_CLEAN"]
    ].head(5)
)

# Verify that all three cleaned columns exist
print("\nCleaned Text Columns:")
for col in text_columns:
    print(col + "_CLEAN")

print("\nTotal Records:", len(df_clean))


In [ ]:

# Step 26: Identify free-text columns for NLP
# HanuAi ML Assessment - Text Mining

import pandas as pd

# Identify unstructured text columns
text_columns = [
    "CUSTOMER_VERBATIM",
    "CAUSAL_VERBATIM",
    "CORRECTION_VERBATIM"
]

# Analyze the completeness and length of each text field
text_summary = []

for col in text_columns:

    # Convert missing values to empty text
    text_data = df_clean[col].fillna("").astype(str)

    # Identify non-empty text entries
    valid_text = text_data.str.strip() != ""

    # Count words in each entry
    word_counts = text_data.str.split().str.len()

    text_summary.append({
        "Column": col,
        "Available Texts": valid_text.sum(),
        "Missing or Blank": (~valid_text).sum(),
        "Average Word Count": round(word_counts.mean(), 2)
    })

# Create a summary table
text_summary_df = pd.DataFrame(text_summary)

print("HANUAI - FREE TEXT DATA ANALYSIS")
display(text_summary_df)

# Display one sample from each text column
for col in text_columns:
    print("\nColumn:", col)

    samples = df_clean[col].dropna()
    samples = samples[samples.astype(str).str.strip() != ""]

    if not samples.empty:
        print("Example:", str(samples.iloc[0])[:250])
    else:
        print("No available text.")


In [ ]:

# Step 25: Manufacturing Plant Complaint Analysis
# HanuAi ML Assessment - Advanced EDA

import matplotlib.pyplot as plt
import pandas as pd

# Count complaints associated with each plant
plant_counts = (
    df_clean["BUILD_PLANT_DESC"]
    .fillna("Unknown")
    .value_counts()
)

# Calculate complaint percentages
plant_summary = pd.DataFrame({
    "Complaint Count": plant_counts,
    "Percentage": (
        plant_counts / len(df_clean) * 100
    ).round(2)
})

# Display summary table
print("MANUFACTURING PLANT COMPLAINT ANALYSIS")
display(plant_summary)

# Create a horizontal bar chart
plt.figure(figsize=(11, 6))

bars = plt.barh(
    plant_counts.index[::-1],
    plant_counts.values[::-1],
    color="steelblue"
)

plt.title("Customer Complaints by Manufacturing Plant")
plt.xlabel("Number of Complaints")
plt.ylabel("Manufacturing Plant")

# Display complaint counts on bars
plt.bar_label(bars, padding=3)

plt.xlim(0, plant_counts.max() * 1.15)
plt.tight_layout()
plt.show()

# Identify the plant with most complaints
print("Plant with Highest Complaints:",
      plant_counts.idxmax())

print("Complaint Count:",
      plant_counts.max())


In [ ]:

# Step 24: Vehicle Model Complaint Analysis
# HanuAi ML Assessment - Advanced EDA

import pandas as pd
import matplotlib.pyplot as plt

# Count complaints for each vehicle model
model_counts = (
    df_clean["MODEL"]
    .fillna("Unknown")
    .value_counts()
)

# Select top 10 models
top_models = model_counts.head(10)

print("TOP 10 VEHICLE MODELS BY COMPLAINT COUNT")
display(top_models)

# Create horizontal bar chart
plt.figure(figsize=(11, 6))

bars = plt.barh(
    top_models.index[::-1],
    top_models.values[::-1],
    color="steelblue"
)

plt.title("Top 10 Vehicle Models by Customer Complaints")
plt.xlabel("Number of Complaints")
plt.ylabel("Vehicle Model")

plt.bar_label(bars, padding=3)
plt.xlim(0, top_models.max() * 1.15)
plt.tight_layout()
plt.show()

# Identify the leading recorded cause for each top model
top_model_data = df_clean[
    df_clean["MODEL"].isin(top_models.index)
].copy()

top_model_data["CAUSAL_CD_DESC"] = (
    top_model_data["CAUSAL_CD_DESC"]
    .fillna("Unknown")
)

cause_by_model = pd.crosstab(
    top_model_data["MODEL"],
    top_model_data["CAUSAL_CD_DESC"]
)

# Find the most frequent recorded cause per model
model_summary = pd.DataFrame({
    "Total Complaints": top_models,
    "Most Frequent Technical Cause":
        cause_by_model.idxmax(axis=1),
    "Cause Count":
        cause_by_model.max(axis=1)
})

print("\nMODEL-WISE TECHNICAL CAUSE SUMMARY")
display(model_summary)


In [ ]:

# Step 23: Technical Failure Cause Analysis
# HanuAi ML Assessment - Advanced EDA

import pandas as pd
import matplotlib.pyplot as plt

# Standardize technical failure category names
causal_categories = (
    df_clean["CAUSAL_CD_DESC"]
    .fillna("Unknown")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .replace("", "Unknown")
)

# Count failures by technical category
causal_counts = causal_categories.value_counts()

# Create frequency and percentage summary
causal_summary = pd.DataFrame({
    "Complaint Count": causal_counts,
    "Percentage": (
        causal_counts / len(df_clean) * 100
    ).round(2)
})

print("TECHNICAL FAILURE CAUSE ANALYSIS")
display(causal_summary)

# Select the top 10 technical failure categories
top_10_causes = causal_counts.head(10).sort_values()

# Create horizontal bar chart
plt.figure(figsize=(13, 7))

bars = plt.barh(
    top_10_causes.index,
    top_10_causes.values,
    color="steelblue"
)

plt.title("Top 10 Technical Failure Categories")
plt.xlabel("Number of Complaints")
plt.ylabel("Technical Failure Cause")

plt.bar_label(bars, padding=3)
plt.xlim(0, top_10_causes.max() * 1.15)

plt.tight_layout()
plt.show()

# Display the most frequent technical cause
print("Most Frequent Technical Cause:",
      causal_counts.idxmax())

print("Number of Complaints:",
      causal_counts.max())


In [ ]:

# Step 22: Analyze customer complaint categories
# HanuAi ML Assessment - Category Analysis

import matplotlib.pyplot as plt

# Standardize spaces in complaint category names
complaint_categories = (
    df_clean["COMPLAINT_CD_DESC"]
    .fillna("Unknown")
    .astype(str)
    .str.replace("\u00a0", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Count complaints in each category
category_counts = complaint_categories.value_counts()

# Display category frequencies
print("CUSTOMER COMPLAINT CATEGORY ANALYSIS")
print("=" * 45)
display(category_counts)

# Create horizontal bar chart
plt.figure(figsize=(12, 6))

category_counts.sort_values().plot(
    kind="barh",
    color="steelblue"
)

plt.title("Most Frequently Reported Complaint Categories")
plt.xlabel("Number of Complaints")
plt.ylabel("Complaint Category")

plt.tight_layout()
plt.show()

# Display the most frequent complaint
print("Most Frequent Category:", category_counts.idxmax())
print("Total Complaints:", category_counts.max())


In [ ]:

# Step 21: Monthly customer complaint trend analysis
# HanuAi ML Assessment - EDA

import matplotlib.pyplot as plt

# Group complaints by month
monthly_complaints = (
    df_clean.groupby(
        df_clean["Opened date"].dt.to_period("M")
    )
    .size()
    .sort_index()
)

# Convert monthly periods into dates for plotting
months = monthly_complaints.index.to_timestamp()

# Create a line chart
plt.figure(figsize=(12, 5))

plt.plot(
    months,
    monthly_complaints.values,
    marker="o",
    linewidth=2,
    color="steelblue"
)

plt.title("Monthly Customer Complaint Trends")
plt.xlabel("Complaint Month")
plt.ylabel("Number of Complaints")

plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Display the chart
plt.show()

# Display the month with the highest complaint count
print("Highest Complaint Month:", monthly_complaints.idxmax())
print("Number of Complaints:", monthly_complaints.max())


In [ ]:

# Step 20: Complaint distribution by vehicle make
# HanuAi ML Assessment - Data Visualization

import matplotlib.pyplot as plt

# Count complaints for each vehicle make
make_counts = df_clean["MAKE"].fillna("Unknown").value_counts()

# Create the bar chart
plt.figure(figsize=(10, 5))

bars = plt.bar(
    make_counts.index,
    make_counts.values,
    color="steelblue"
)

# Add title and axis labels
plt.title("Customer Complaints by Vehicle Make", fontsize=14)
plt.xlabel("Vehicle Make")
plt.ylabel("Number of Complaints")

# Display complaint counts above the bars
plt.bar_label(bars, padding=3)

# Improve readability
plt.xticks(rotation=20, ha="right")
plt.tight_layout()

# Display the chart
plt.show()


In [ ]:

# Step 19: Statistical Analysis
# HanuAi ML Assessment

# Calculate days between vehicle build and complaint registration
df_clean["Days_Since_Build"] = (
    df_clean["Opened date"] - df_clean["BUILD_DATE"]
).dt.days

# Select numerical features
numeric_columns = ["MODLYR", "Days_Since_Build"]

# Calculate descriptive statistics
statistics = df_clean[numeric_columns].agg([
    "count", "mean", "median", "std", "min", "max"
]).T.round(2)

print("HANUAI - STATISTICAL ANALYSIS")
display(statistics)

# Check for invalid negative durations
negative_days = (df_clean["Days_Since_Build"] < 0).sum()
print("Negative days since build:", negative_days)

# Analyze complaint counts by vehicle make
print("\nCOMPLAINT FREQUENCY BY VEHICLE MAKE")
display(df_clean["MAKE"].value_counts())


In [ ]:

# Step 18: Identify critical columns for business analysis

# Select columns important for stakeholder insights
critical_columns = [
    "Event id",
    "Opened date",
    "CUSTOMER_VERBATIM",
    "CAUSAL_VERBATIM",
    "CORRECTION_VERBATIM",
    "COMPLAINT_CD_DESC",
    "CAUSAL_CD_DESC",
    "MAKE",
    "MODEL",
    "MODLYR",
    "BUILD_DATE",
    "IN_USE_DATE",
    "BUILD_PLANT_DESC"
]

# Create a summary of important columns
critical_summary = pd.DataFrame({
    "Column": critical_columns,
    "Data Type": [
        str(df_clean[col].dtype)
        for col in critical_columns
    ],
    "Missing Values": [
        df_clean[col].isna().sum()
        for col in critical_columns
    ],
    "Unique Values": [
        df_clean[col].nunique()
        for col in critical_columns
    ]
})

print("HANUAI - CRITICAL COLUMNS ANALYSIS")
print("=" * 45)

display(critical_summary)


In [ ]:

# Step 17: Clean and standardize date columns
# HanuAi ML Assessment - Date Preprocessing

# Convert complaint opened date
df_clean["Opened date"] = pd.to_datetime(
    df_clean["Opened date"],
    format="%m/%d/%Y",
    errors="coerce"
)

# Convert vehicle build date
df_clean["BUILD_DATE"] = pd.to_datetime(
    df_clean["BUILD_DATE"],
    format="%Y-%m-%dT%H:%M:%S.%fZ",
    errors="coerce"
)

# Convert vehicle in-use date
df_clean["IN_USE_DATE"] = pd.to_datetime(
    df_clean["IN_USE_DATE"],
    format="%Y-%m-%dT%H:%M:%S.%fZ",
    errors="coerce"
)

# Check the updated date columns
print("DATE COLUMN DATA TYPES")
print(df_clean[
    ["Opened date", "BUILD_DATE", "IN_USE_DATE"]
].dtypes)

print("\nMISSING OR INVALID DATES")
print(df_clean[
    ["Opened date", "BUILD_DATE", "IN_USE_DATE"]
].isna().sum())

print("\nFIRST FIVE RECORDS")
display(df_clean[
    ["Opened date", "BUILD_DATE", "IN_USE_DATE"]
].head())


In [ ]:

# Step 16: Identify duplicate records
# HanuAi ML Assessment - Data Quality Check

print("HANUAI DATASET - DUPLICATE ANALYSIS")
print("=" * 45)

# Count completely identical rows
exact_duplicates = df_clean.duplicated().sum()

# Count repeated Event IDs
duplicate_ids = df_clean["Event id"].duplicated().sum()

# Display duplicate summary
print("Total Records:", len(df_clean))
print("Exact Duplicate Rows:", exact_duplicates)
print("Duplicate Event IDs:", duplicate_ids)

# Display duplicate Event IDs if any exist
if duplicate_ids > 0:
    print("\nDuplicate Event ID Records:")
    display(
        df_clean[
            df_clean.duplicated(
                subset=["Event id"], keep=False
            )
        ]
    )
else:
    print("\nNo duplicate Event IDs found.")


In [ ]:

# Step 15: Handle missing values
# HanuAi ML Assessment - Data Cleaning

# Create a copy to preserve the original raw dataset
df_clean = df.copy()

# Fill missing technical descriptions with empty text
df_clean["CAUSAL_VERBATIM"] = (
    df_clean["CAUSAL_VERBATIM"].fillna("")
)

# Label missing causal categories as Unknown
df_clean["CAUSAL_CD_DESC"] = (
    df_clean["CAUSAL_CD_DESC"].fillna("Unknown")
)

# Keep missing dates unchanged rather than inventing dates
# Preserve sparse entity-tag columns for NLP extraction later

# Check the results
print("DATA CLEANING SUMMARY")

print("Original records:", len(df))
print("Cleaned records:", len(df_clean))

print("\nRemaining missing values:")
print(df_clean.isnull().sum())


In [ ]:

# Step 14: Identify missing values in the dataset

# Count missing values in every column
missing_values = df.isnull().sum()

# Calculate the percentage of missing values
missing_percentage = (df.isnull().mean() * 100).round(2)

# Create a summary table
missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage
})

# Sort columns by highest number of missing values
missing_summary = missing_summary.sort_values(
    by="Missing Values",
    ascending=False
)

# Display results
print("HANUAI DATASET - MISSING VALUES ANALYSIS")
display(missing_summary)


In [ ]:

# Step 13: Understand the dataset structure

# Display dataset information
print("HANUAI DATASET INFORMATION")
print("=" * 40)

df.info()


In [ ]:

# Task 2: Load and inspect the HanuAi dataset

import pandas as pd
from pathlib import Path

# Find the original HanuAi CSV file
csv_files = list(Path(".").glob("Hanuai_task2_assignment*.csv"))

# Verify that exactly one matching file exists
assert len(csv_files) == 1, f"Expected one dataset, found: {csv_files}"

# Load the CSV file into a Pandas DataFrame
df = pd.read_csv(csv_files[0])

# Display the number of rows and columns
print("Dataset Shape:", df.shape)

# Display the first five records
display(df.head())


In [ ]:

# HanuAi ML Assessment - Environment Verification

import sys
import pandas as pd

print("Python Version:", sys.version.split()[0])
print("Pandas Version:", pd.__version__)
print("Python Environment:", sys.executable)
print("HanuAi ML Assessment setup successful!")
